In [1]:
%%html
<link href="https://fonts.googleapis.com/css2?family=Source+Serif+4:wght@300;400;500;600;700&display=swap" rel="stylesheet">

<style>
/* markdown + output */
body, .markdown-body, .jp-RenderedHTMLCommon, div.text_cell_render,
.notebook, .notebook * {
  font-family: "Source Serif 4", serif !important;
}

/* code editor (monaco) */
.monaco-editor, .monaco-editor * {
  font-family: "Source Serif 4", serif !important;
}
</style>

### interval estimation for mean

interval estimation helps us find a range where the true population parameter likely exists based on sample data. instead of giving a single point estimate, we provide a confidence interval.

key components:
- sample mean (x̄) - calculated from sample data
- confidence level (cl) - typically 95% or 99%
- margin of error - depends on standard error and critical value
- critical value - from z or t distribution based on confidence level

formula for confidence interval:
- ci = x̄ ± z * (σ / √n) when population std is known (z-distribution)
- ci = x̄ ± t * (s / √n) when population std is unknown (t-distribution)

interpretation: we are 95% confident that the true population mean lies within this calculated interval.

In [ ]:
import numpy as np
import pandas as pd
import scipy
from scipy import stats
from scipy.stats import norm, t, binom, poisson, expon, chi2, f
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
from statsmodels import stats as sm_stats
from statsmodels.stats import weightstats as sswa
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
os.chdir(r'/Users/proxim/Desktop/CDAC-DBDA-coursework/08.advanced-analytics-stats')

### example: confidence interval for mean

given:
- sample size n = 100
- sample mean x̄ = 45.7
- population std σ = 6.8
- confidence level = 95%

we want to find the interval estimate where the true population mean likely lies.

In [ ]:
# method 1: using z-distribution (manual calculation)
n = 100
sample_mean = 45.7
pop_std = 6.8
cl = 0.95

# critical values for 95% confidence (2.5% in each tail)
z_lower = norm.ppf(0.025)  # -1.96
z_upper = norm.ppf(0.975)  # +1.96

# calculate confidence interval bounds
lower_bound = sample_mean + z_lower * pop_std / np.sqrt(n)
upper_bound = sample_mean + z_upper * pop_std / np.sqrt(n)

print(f'95% confidence interval using z-distribution:')
print(f'lower bound = {lower_bound:.2f}')
print(f'upper bound = {upper_bound:.2f}')
print(f'\ninterpretation: we are 95% confident that true mean is between {lower_bound:.2f} and {upper_bound:.2f}')

In [ ]:
# method 2: using t-distribution (more accurate for smaller samples)
# critical values from t-distribution with df = n-1
t_lower = t.ppf((1-cl)/2, n-1)
t_upper = t.ppf(cl + (1-cl)/2, n-1)

lower_t = sample_mean + t_lower * pop_std / np.sqrt(n)
upper_t = sample_mean + t_upper * pop_std / np.sqrt(n)

print(f'95% confidence interval using t-distribution:')
print(f'lower bound = {lower_t:.2f}')
print(f'upper bound = {upper_t:.2f}')

In [ ]:
# method 3: using built-in scipy function
ci = scipy.stats.t.interval(cl, n-1, sample_mean, pop_std/np.sqrt(n))
print(f'95% confidence interval using scipy:')
print(f'interval = {ci}')
print(f'\nconclusion: 95% chance that actual population mean lies between {ci[0]:.2f} and {ci[1]:.2f}')
print(f'2.5% chance it may be less than {ci[0]:.2f}')
print(f'2.5% chance it may be greater than {ci[1]:.2f}')

### interval estimation for poisson

for discrete count data following poisson distribution, we can also find confidence intervals. poisson distribution is used when counting occurrences of events (like number of calls per hour, defects per product, etc).

In [ ]:
# example: average 20 events occur, find 95% confidence interval
lambda_param = 20
poisson_ci = scipy.stats.poisson.interval(0.95, lambda_param)
print(f'95% confidence interval for poisson (λ=20):')
print(f'interval = {poisson_ci}')
print(f'\ninterpretation: we expect between {poisson_ci[0]} and {poisson_ci[1]} events with 95% confidence')

In [ ]:
# finding bounds using ppf (percent point function)
lower_poisson = poisson.ppf(0.025, 20)
upper_poisson = poisson.ppf(0.975, 20)
print(f'using ppf method:')
print(f'lower bound = {lower_poisson}')
print(f'upper bound = {upper_poisson}')

### interval estimation for proportion

when working with proportions or percentages (like success rate, approval rating, conversion rate), we estimate the confidence interval for the true population proportion.

formula: p ± z * √(p(1-p)/n)

where:
- p = sample proportion
- n = sample size
- z = critical value from standard normal distribution

In [ ]:
# example: out of 500 people surveyed, 200 responded positively
n = 500
r = 200  # number of positive responses
p = r / n  # sample proportion
sd = np.sqrt(p * (1 - p))  # standard deviation for proportion

# for 95% confidence level
z_lower = norm.ppf(0.025)
z_upper = norm.ppf(0.975)

# calculate bounds
lower_prop = p + z_lower * sd / np.sqrt(n)
upper_prop = p + z_upper * sd / np.sqrt(n)

print(f'sample proportion = {p:.3f} or {p*100:.1f}%')
print(f'\n95% confidence interval for proportion:')
print(f'lower bound = {lower_prop:.3f} ({lower_prop*100:.1f}%)')
print(f'upper bound = {upper_prop:.3f} ({upper_prop*100:.1f}%)')
print(f'\ninterpretation: true population proportion is between {lower_prop*100:.1f}% and {upper_prop*100:.1f}% with 95% confidence')

### hypothesis testing - introduction

hypothesis testing is a statistical method to make decisions about population parameters based on sample data.

key concepts:
- null hypothesis (H₀): the claim we assume to be true (status quo)
- alternative hypothesis (H₁): what we want to prove
- p-value: probability of observing the data if H₀ is true
- significance level (α): threshold for decision, typically 0.05
- decision rule: if p-value < α, reject H₀; otherwise, fail to reject H₀

types of tests:
- left tail: H₁ states parameter is less than claim
- right tail: H₁ states parameter is greater than claim
- two tail: H₁ states parameter is not equal to claim

### one-sample z-test / one-sample t-test

purpose: compare sample mean against a claimed value and determine if the difference is statistically significant.

when to use:
- z-test: large sample (n ≥ 30) or known population σ
- t-test: small sample (n < 30) with unknown population σ

formulas:
- z = (x̄ - μ₀) / (σ / √n)
- t = (x̄ - μ₀) / (s / √n)

where:
- x̄ = sample mean
- μ₀ = claimed population mean
- σ = population standard deviation (z-test)
- s = sample standard deviation (t-test)
- n = sample size

In [ ]:
# load data for hypothesis testing examples
df = pd.read_excel('data/CDAC_DataBook.xlsx', sheet_name='faithful')
df.head()

In [ ]:
erupt = df.eruptions
print(f'sample size = {len(erupt)}')
print(f'sample mean = {np.mean(erupt):.2f}')
print(f'sample std = {np.std(erupt, ddof=1):.2f}')

### example: left tail test

claim: population mean eruption time is at least 3.6 minutes

hypotheses:
- H₀: μ ≥ 3.6 (claim - population mean is at least 3.6)
- H₁: μ < 3.6 (alternative - we want to prove mean is less than 3.6)

this is a left tail test because H₁ has < symbol.
significance level α = 0.05 (confidence level = 95%)

In [ ]:
# perform one-sample z-test
z_stat, p_value = sswa.ztest(erupt, value=3.6, alternative='smaller')

print('one-sample z-test results:')
print(f'z-statistic = {z_stat:.4f}')
print(f'p-value = {p_value:.4f}')
print(f'\ndecision:')
if p_value < 0.05:
    print('reject H₀: evidence that mean < 3.6')
else:
    print('fail to reject H₀: insufficient evidence that mean < 3.6')
    
print(f'\ninterpretation:')
print(f'p-value ({p_value:.4f}) tells us the probability of observing this sample')
print(f'mean if true population mean is actually 3.6')
print(f'since p-value > 0.05, difference can be attributed to chance variation')

In [ ]:
# manual calculation of z-statistic
sample_mean = np.mean(erupt)
claim_value = 3.6
sample_std = np.std(erupt, ddof=1)
n = len(erupt)

# formula: z = (x̄ - μ₀) / (s / √n)
z_manual = (sample_mean - claim_value) / (sample_std / np.sqrt(n))
print(f'manual calculation:')
print(f'z-statistic = ({sample_mean:.2f} - {claim_value}) / ({sample_std:.2f} / √{n})')
print(f'z-statistic = {z_manual:.4f}')

In [ ]:
# finding critical value for left tail test (α = 0.05)
t_crit = t.ppf(0.05, n-1)
print(f't-critical value (df={n-1}) = {t_crit:.4f}')
print(f'\ndecision rule: if t-statistic < {t_crit:.4f}, reject H₀')
print(f'our t-statistic ({z_manual:.4f}) is not less than {t_crit:.4f}')
print(f'therefore, we fail to reject H₀')

In [ ]:
# calculating 95% upper confidence bound (one-sided)
# for left tail test, we calculate upper bound
# formula: x̄ + t_critical * (s / √n)

t_crit_positive = t.ppf(0.95, n-1)  # for upper bound
upper_bound = np.mean(erupt) + t_crit_positive * np.std(erupt, ddof=1) / np.sqrt(n)

print(f'95% upper confidence bound = {upper_bound:.4f}')
print(f'sample mean = {np.mean(erupt):.4f}')
print(f'claim value = 3.6')
print(f'\ninterpretation:')
print(f'we are 95% confident that true mean ≤ {upper_bound:.4f}')
print(f'since {upper_bound:.4f} > 3.6, the claim (mean ≥ 3.6) is reasonable')

### example: right tail test

claim: maximum eruption time is 3.42 minutes

hypotheses:
- H₀: μ ≤ 3.42 (claim - mean is at most 3.42)
- H₁: μ > 3.42 (alternative - mean is greater than 3.42)

this is a right tail test because H₁ has > symbol.

In [ ]:
# perform right tail test
z_stat_right, p_val_right = sswa.ztest(erupt, value=3.42, alternative='larger')

print('right tail test results:')
print(f'z-statistic = {z_stat_right:.4f}')
print(f'p-value = {p_val_right:.4f}')
print(f'\ndecision:')
if p_val_right < 0.05:
    print('reject H₀: evidence that mean > 3.42')
else:
    print('fail to reject H₀: insufficient evidence that mean > 3.42')

In [ ]:
# calculating 95% lower confidence bound (one-sided)
# for right tail test, we calculate lower bound
# formula: x̄ - t_critical * (s / √n)

t_crit_positive = t.ppf(0.95, n-1)
lower_bound = np.mean(erupt) - t_crit_positive * np.std(erupt, ddof=1) / np.sqrt(n)

print(f'95% lower confidence bound = {lower_bound:.4f}')
print(f'sample mean = {np.mean(erupt):.4f}')
print(f'claim value = 3.42')
print(f'\ninterpretation:')
print(f'we are 95% confident that true mean ≥ {lower_bound:.4f}')
print(f'since {lower_bound:.4f} > 3.42, there is evidence mean > 3.42')

### practice: hypothesis test with small sample

claim: population mean > 6

hypotheses:
- H₀: μ ≤ 6
- H₁: μ > 6 (right tail test)

In [ ]:
my_list = [3, 6, 5, 4, 6, 7, 9, 6, 2]
my_array = np.array(my_list)

print(f'sample size = {len(my_list)}')
print(f'sample mean = {np.mean(my_list):.4f}')
print(f'sample std = {np.std(my_list, ddof=1):.4f}')

In [ ]:
# perform right tail test
z_stat, p_val = sswa.ztest(my_list, value=6, alternative='larger')

print('test results:')
print(f'z-statistic = {z_stat:.4f}')
print(f'p-value = {p_val:.4f}')
print(f'\ndecision: fail to reject H₀ (p-value > 0.05)')
print(f'conclusion: insufficient evidence that mean > 6')

In [ ]:
# manual t-statistic calculation
t_manual = (np.mean(my_list) - 6) / (np.std(my_list, ddof=1) / np.sqrt(len(my_list)))
print(f't-statistic = {t_manual:.4f}')
print(f'\ninterpretation: sample mean is {t_manual:.4f} standard errors away from claimed value')

In [ ]:
# calculate p-value using t-distribution
p_t = 1 - t.cdf(t_manual, len(my_list) - 1)  # right tail
print(f'p-value from t-distribution = {p_t:.4f}')
print(f'\nsince p-value > 0.05, we fail to reject H₀')
print(f'the sample does not provide strong evidence that mean > 6')

### two-sample z-test / two-sample t-test

purpose: compare means of two independent samples and determine if their difference is statistically significant.

formula for test statistic:
- z = (x̄₁ - x̄₂ - d₀) / SE
- where SE = √(σ₁²/n₁ + σ₂²/n₂)

for t-test:
- t = (x̄₁ - x̄₂ - d₀) / SE
- where SE = √(s₁²/n₁ + s₂²/n₂)

d₀ is the claimed difference (often 0)

use cases:
- compare treatment vs control groups
- compare before vs after
- compare two populations

### example: comparing two groups

claim: average weight difference between males and females is at least 7 kg

hypotheses:
- H₀: μ_males - μ_females ≥ 7
- H₁: μ_males - μ_females < 7 (left tail test)

In [ ]:
males = [76, 56, 62, 65, 54, 70]
females = [54, 57, 48, 64, 57]

print(f'males:')
print(f'  n = {len(males)}')
print(f'  mean = {np.mean(males):.2f} kg')
print(f'  std = {np.std(males, ddof=1):.2f} kg')
print(f'\nfemales:')
print(f'  n = {len(females)}')
print(f'  mean = {np.mean(females):.2f} kg')
print(f'  std = {np.std(females, ddof=1):.2f} kg')
print(f'\nobserved difference = {np.mean(males) - np.mean(females):.2f} kg')

In [ ]:
# perform two-sample z-test
z_stat_2samp, p_val_2samp = sswa.ztest(males, females, value=7, alternative='smaller')

print('two-sample z-test results:')
print(f'z-statistic = {z_stat_2samp:.4f}')
print(f'p-value = {p_val_2samp:.4f}')
print(f'\ndecision:')
if p_val_2samp < 0.05:
    print('reject H₀: evidence that difference < 7 kg')
else:
    print('fail to reject H₀: claim that difference ≥ 7 kg is reasonable')

In [ ]:
# visualization: comparing two groups
plt.figure(figsize=(12, 5))

# boxplot comparison
plt.subplot(1, 2, 1)
plt.boxplot([males, females], labels=['males', 'females'], 
            patch_artist=True, boxprops=dict(facecolor='lightblue'))
plt.ylabel('weight (kg)')
plt.title('weight distribution comparison')
plt.grid(True, alpha=0.3)

# histogram comparison
plt.subplot(1, 2, 2)
plt.hist(males, alpha=0.5, label='males', bins=5, edgecolor='black', color='blue')
plt.hist(females, alpha=0.5, label='females', bins=5, edgecolor='black', color='pink')
plt.xlabel('weight (kg)')
plt.ylabel('frequency')
plt.title('weight distribution')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### practice problems - hypothesis testing

solve the following hypothesis testing problems using appropriate tests.

In [ ]:
# q1. msrtc claims max time to reach kolhapur is 5 hours

# hypotheses:
# H₀: μ ≤ 5 (claim - travel time is at most 5 hours)
# H₁: μ > 5 (right tail test)

# example data (hours)
trip = [4.8, 5.2, 6.1, 4.9, 5.5, 5.8, 4.7, 5.3, 6.0, 5.1]

# perform test
z_stat_q1, p_val_q1 = sswa.ztest(trip, value=5, alternative='larger')
print('q1: msrtc travel time test')
print(f'sample mean = {np.mean(trip):.2f} hours')
print(f'sample std = {np.std(trip, ddof=1):.2f}')
print(f'z-statistic = {z_stat_q1:.4f}')
print(f'p-value = {p_val_q1:.4f}')

# calculate lower bound (one-sided 95%)
alpha = 0.05
t_crit_q1 = t.ppf(1 - alpha, len(trip) - 1)
lower_bound_q1 = np.mean(trip) - t_crit_q1 * np.std(trip, ddof=1) / np.sqrt(len(trip))
print(f'95% lower bound = {lower_bound_q1:.2f} hours')
print(f'decision: lower_bound ({lower_bound_q1:.2f}) > 5? {lower_bound_q1 > 5}')

if p_val_q1 < 0.05:
    print('reject H₀: evidence that mean > 5 hours')
else:
    print('fail to reject H₀: claim is reasonable')
print()

In [ ]:
# q2. car manufacturer claims minimum mileage is 18 kmpl

# hypotheses:
# H₀: μ ≥ 18 (claim - mileage at least 18 kmpl)
# H₁: μ < 18 (left tail test)

# example data (kmpl)
mileage = [17.6, 18.2, 17.9, 18.0, 17.4, 17.8, 18.1, 17.5, 17.7, 18.3]

# perform test
z_stat_q2, p_val_q2 = sswa.ztest(mileage, value=18, alternative='smaller')
print('q2: car mileage test')
print(f'sample mean = {np.mean(mileage):.2f} kmpl')
print(f'sample std = {np.std(mileage, ddof=1):.2f}')
print(f'z-statistic = {z_stat_q2:.4f}')
print(f'p-value = {p_val_q2:.4f}')

# calculate upper bound (one-sided 95%)
t_crit_q2 = t.ppf(1 - alpha, len(mileage) - 1)
upper_bound_q2 = np.mean(mileage) + t_crit_q2 * np.std(mileage, ddof=1) / np.sqrt(len(mileage))
print(f'95% upper bound = {upper_bound_q2:.2f} kmpl')
print(f'decision: upper_bound ({upper_bound_q2:.2f}) < 18? {upper_bound_q2 < 18}')

if p_val_q2 < 0.05:
    print('reject H₀: evidence that mean < 18 kmpl')
else:
    print('fail to reject H₀: claim is reasonable')
print()

In [ ]:
# q3. msrtc claims difference with ksrtc is not more than 1 hour

# hypotheses:
# H₀: μ_msrtc - μ_ksrtc ≤ 1 (claim - difference at most 1 hour)
# H₁: μ_msrtc - μ_ksrtc > 1 (right tail test)

# example data (hours)
msrtc = [4.2, 3.8, 5.0, 4.5, 4.1, 4.8]
ksrtc = [3.1, 3.4, 3.6, 3.9, 3.2, 3.5]

# perform test
z_stat_q3, p_val_q3 = sswa.ztest(msrtc, ksrtc, value=1, alternative='larger')
diff_q3 = np.mean(msrtc) - np.mean(ksrtc)
print('q3: msrtc vs ksrtc travel time difference')
print(f'msrtc mean = {np.mean(msrtc):.2f} hours')
print(f'ksrtc mean = {np.mean(ksrtc):.2f} hours')
print(f'observed difference = {diff_q3:.2f} hours')
print(f'claimed max difference = 1 hour')
print(f'z-statistic = {z_stat_q3:.4f}')
print(f'p-value = {p_val_q3:.4f}')

if p_val_q3 < 0.05:
    print('reject H₀: evidence that difference > 1 hour')
else:
    print('fail to reject H₀: claim is reasonable')
print()

In [ ]:
# q4. instructor claims students get minimum 10k more salary than another batch

# hypotheses:
# H₀: μ_batch1 - μ_batch2 ≥ 10 (claim - difference at least 10k)
# H₁: μ_batch1 - μ_batch2 < 10 (left tail test)

# example data (salaries in thousands)
batch1 = [52, 54, 56, 55, 53, 57, 54, 56]
batch2 = [50, 51, 49, 50, 48, 51, 50, 49]

# perform test
z_stat_q4, p_val_q4 = sswa.ztest(batch1, batch2, value=10, alternative='smaller')
diff_q4 = np.mean(batch1) - np.mean(batch2)
print('q4: batch salary comparison')
print(f'batch1 mean = {np.mean(batch1):.2f}k')
print(f'batch2 mean = {np.mean(batch2):.2f}k')
print(f'observed difference = {diff_q4:.2f}k')
print(f'claimed min difference = 10k')
print(f'z-statistic = {z_stat_q4:.4f}')
print(f'p-value = {p_val_q4:.4f}')

if p_val_q4 < 0.05:
    print('reject H₀: evidence that difference < 10k (claim not supported)')
else:
    print('fail to reject H₀: claim could be true')